In [13]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from pathlib import Path
import leidenalg
import scanpy as sc
import anndata

import torch
import torch.nn as nn

In [16]:
# extract features using different stain generalisation methods and foundation models
model_stains = ['conch_aug', 'gigapath_aug', 'iBOT_aug', 'EXAONEPath_aug', 'UNI_aug',
                'conch_rein', 'gigapath_rein', 'UNI_rein', 'iBOT_rein', 'EXAONEPath_rein',
                'conch_orig', 'EXAONEPath_orig', 'UNI_orig', 'gigapath_orig', 'iBOT_orig']
cohorts = ['KHP_RM', 'KHP_RRM', 'BRACS']

for model_stain in model_stains:
    for cohort in cohorts:
        print(cohort)
        FEATURES = Path(f'/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/{cohort}')
        save_pt = Path(f'/scratch_tmp/prj/cb_normalbreast/prj_NBTPhenotyping/RESULTS/FoundationModel_features/{cohort}_{model_stain}_features.csv')
    
        all_data = []
        counter = 0
        
        for feature_pt in FEATURES.glob(f'**/*_{model_stain}*.h5'):
            try:
                # Load the embeddings and patch_id from the .h5 file
                print(feature_pt)
                with h5py.File(feature_pt, 'r') as f:
                    embedding = np.array(f['embeddings'])
                    img_id = np.array(f['patch_id'])
                print(embedding.shape)
                
                # Decode the patch IDs and create a DataFrame
                img_id = [i.decode('utf-8') for i in img_id]
                fea_df = pd.DataFrame(embedding)
                fea_df.columns = ['embedding_' + str(i) for i in fea_df.columns]
                fea_df['patch_id'] = img_id
                
                # Extract the WSI ID and find the corresponding CSV file
                if cohort == 'NKI':
                    wsi_id = fea_df['patch_id'][0].split(' HE')[0]
                else:
                    wsi_id = '_'.join(fea_df['patch_id'][0].split('_')[:3])
                    
                csv_pt = glob.glob(f'{FEATURES}/{wsi_id}*/{wsi_id}_patch.csv')
                
                if csv_pt:
                    print(csv_pt)
                    counter += 1
                    # Read the corresponding CSV metadata file
                    csv_df = pd.read_csv(csv_pt[0])
                    csv_df['cohort'] = cohort
                    
                    # Merge with the embeddings DataFrame
                    df_merge = pd.merge(csv_df, fea_df, on='patch_id')
                    df_merge.index = df_merge['patch_id']
                    
                    # Append the merged DataFrame to the list
                    all_data.append(df_merge)
            except:
                pass
        
        if all_data:
            final_df = pd.concat(all_data, axis=0)
            save_pt = save_pt.with_name(f'{save_pt.stem}{counter}{save_pt.suffix}')
            final_df.to_csv(save_pt)
            print(save_pt)

KHP_RM
/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17063451_FPE_4/17063451_FPE_4_bagFeature_EXAONEPath_orig.h5
(732, 768)
['/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17063451_FPE_4/17063451_FPE_4_patch.csv']
/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17063838_FPE_4/17063838_FPE_4_bagFeature_EXAONEPath_orig.h5
(3986, 768)
['/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17063838_FPE_4/17063838_FPE_4_patch.csv']
/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17064241_FPE_4/17064241_FPE_4_bagFeature_EXAONEPath_orig.h5
(125, 768)
['/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17064241_FPE_4/17064241_FPE_4_patch.csv']
/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17064241_FPE_1/17064241_FPE_1_bagFeature_EXAONEPath_orig.h5
(70, 768)
['/scratch_tmp/prj/cb_normalbreast/prj_BreastAgeNet/FEATURES/KHP_RM/17064241_FPE_1/17064241_FPE_1_patch.csv']
/scra